# **Custom CNN: Baseline Model Development**

A custom Convolutional Neural Network was developed to establish a baseline for comparison against pretrained models. This notebook documents the architecture design, training procedure, and performance metrics.

### **Design Rationale**
- Gradual capacity increase: Filter counts (32 → 64 → 128) allow the network to learn increasingly abstract features
- Dropout schedule: Higher dropout in the dense layers (50%) reflects the greater overfitting risk in fully connected stages

### Imports

In [2]:
import tensorflow as tf
#prevent TF from grabbing all GPU memory at once
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [3]:
from tensorflow.keras import mixed_precision

# Tell Keras to use float16 for memory, but keep float32 for numeric stability in the loss
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print('Compute dtype: %s' % policy.compute_dtype)
print('Variable dtype: %s' % policy.variable_dtype)

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6
Compute dtype: float16
Variable dtype: float32


In [4]:
import os
import sys
import keras.applications
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from keras.applications.efficientnet import preprocess_input


if os.getcwd().endswith('models'):
    os.chdir('..')
    
from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [5]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TF built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available to TF:", tf.test.is_gpu_available())  # deprecated but still works

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: True
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU available to TF: True


In [1]:
# COLAB

from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/MDSAA-DS/DL-project/dataverse_files/data"

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import json
from PIL import Image
import matplotlib.pyplot as plt

if os.getcwd().endswith('models'):
    os.chdir('..')

# from utils.utils_model import *

with open(os.path.join(base_path, "label2idx.json"), "r") as f:
    label2idx = json.load(f)

In [6]:
# Load label mapping first — everything else depends on it
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

N_CLASSES  = len(label2idx)
BATCH_SIZE = 32

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

# Normalise column names and encode labels for all splits
for df in [train_df, val_df, test_df]:
    if 'cleaned_path' in df.columns and 'image_path' in df.columns:
        df.drop(columns=['image_path'], inplace=True)
    df.rename(columns={'cleaned_path': 'image_path'}, inplace=True)
    df['dx_encoded'] = df['dx'].map(label2idx).astype(int)

# Build datasets
train_ds = make_dataset(train_df, shuffle=True, repeat=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [7]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

Steps per epoch: 394
Class weights: {0: 1.4493569131832797, 1: 1.2340862422997947, 2: 0.8370473537604457, 3: 2.504166666666667, 4: 0.8727008712487899, 5: 0.42867332382310985, 6: 2.3415584415584414}


### Data Configuration

In [4]:
# COLAB

train_path = os.path.join(base_path, "augmented_metadata.csv")
val_path = os.path.join(base_path, "val_split.csv")
test_path = os.path.join(base_path, "test_split.csv")

train_df = pd.read_csv(train_path, sep=",")[['image_id', 'dataset', 'lesion_id','image_path', 'dx', 'dx_encoded']]
val_df = pd.read_csv(val_path, sep=",")[['image_id', 'dataset', 'lesion_id','image_path', 'dx_encoded']]
test_df = pd.read_csv(test_path, sep=",")[['image_id', 'dataset', 'lesion_id','image_path', 'dx_encoded']]

val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_df['dx_encoded'] = train_df['dx'].map(label2idx).astype(int)

train_df['image_path'] = train_df['image_path'].apply(lambda x: base_path + x[6:].replace('\\', '/'))
val_df['image_path'] = val_df['image_path'].apply(lambda x: base_path + x[6:].replace('\\', '/'))
test_df['image_path'] = test_df['image_path'].apply(lambda x: base_path + x[6:].replace('\\', '/'))

In [ ]:
# Load the CSVs
train_df = pd.read_csv('data/augmented_metadata.csv')[['image_id', 'dataset', 'lesion_id','image_path', 'dx', 'dx_encoded']]
val_df = pd.read_csv('data/val_split.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path', 'dx_encoded']]
test_df = pd.read_csv('data/test_split.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path','dx_encoded']]

val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_df['dx_encoded'] = train_df['dx'].map(label2idx).astype(int)

In [5]:
N_CLASSES  = len(np.unique(train_df.dx_encoded))
BATCH_SIZE = 128

# Build datasets
train_ds = make_dataset_new(train_df, shuffle=True, repeat=True, batch_size=BATCH_SIZE)
val_ds   = make_dataset_new(val_df, batch_size=BATCH_SIZE)
test_ds  = make_dataset_new(test_df, batch_size=BATCH_SIZE)

STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [6]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

Steps per epoch: 98
Class weights: {0: np.float64(1.4493569131832797), 1: np.float64(1.2340862422997947), 2: np.float64(0.8370473537604457), 3: np.float64(2.504166666666667), 4: np.float64(0.8727008712487899), 5: np.float64(0.42867332382310985), 6: np.float64(2.3415584415584414)}


### Model Configuration

In [8]:
def build_model():
    model = keras.Sequential([
        keras.layers.Conv2D(32, 3, activation="relu", padding="same", input_shape=(224, 224, 3)),
        keras.layers.MaxPooling2D(2),
        keras.layers.Dropout(0.25),

        keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Dropout(0.25),

        keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Dropout(0.3),

        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(N_CLASSES)  # NO softmax!
    ])
    return model

In [16]:
# 1. Build the custom model
custom_model = build_model()
custom_model.summary()

# 2. Compile (Standard LR for from-scratch training)
custom_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)]
)

# 3. Callbacks (Give it a bit more patience since it learns from scratch)
my_callbacks = get_callbacks(
    checkpoint_path="checkpoints/model_CUSTOM_best.weights.h5",
    model=custom_model,    
    max_diff=0.15,       
    patience_es=15, 
    patience_lr=5
)

# 4. Train the whole network at once!
history_custom = custom_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=my_callbacks
)

# 5. Evaluate
plot_history(history_custom, "Model Custom CNN — Training")
results_custom = evaluate_model(custom_model, test_ds, label2idx, "Model Custom CNN")

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_9 (Conv2D)           (None, 224, 224, 32)      896       
                                                                 
 max_pooling2d_9 (MaxPooling  (None, 112, 112, 32)     0         
 2D)                                                             
                                                                 
 dropout_12 (Dropout)        (None, 112, 112, 32)      0         
                                                                 
 conv2d_10 (Conv2D)          (None, 112, 112, 64)      18496     
                                                                 
 max_pooling2d_10 (MaxPoolin  (None, 56, 56, 64)       0         
 g2D)                                                            
                                                                 
 dropout_13 (Dropout)        (None, 56, 56, 64)       

KeyboardInterrupt: 